**Steps**

1. Load the fixed study reference.
2. Load MIMIC-IV demographic information.
3. Load selected laboratory measurements.
4. Aggregate laboratory values for each hospital admission.
5. Merge the structured features with the study labels.
6. Handle missing values.
7. Encode categorical variables.
8. Standardise numerical features.
9. Save the processed structured dataset.

**Output:** structured_processed.csv

In [1]:
# Setup
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import StandardScaler

base_path = '/content/drive/MyDrive/dissertation_project/data'
raw_path = f'{base_path}/raw'
processed_path = f'{base_path}/processed'

os.makedirs(raw_path, exist_ok=True)
os.makedirs(processed_path, exist_ok=True)

print("Directories ready.")

Mounted at /content/drive
Directories ready.


In [2]:
# Load the fixed sample
reference = pd.read_csv(
    f'{processed_path}/1_structured_reference.csv'
)

print("Reference shape:", reference.shape)
print("\nColumns:")
print(reference.columns.tolist())

reference.head()

Reference shape: (2200, 9)

Columns:
['subject_id', 'study_id', 'No Finding', 'Support Devices', 'Pleural Effusion', 'Lung Opacity', 'Atelectasis', 'Cardiomegaly', 'Edema']


,subject_id,study_id,No Finding,Support Devices,Pleural Effusion,Lung Opacity,Atelectasis,Cardiomegaly,Edema
0,19963242,51118747,1.0,NaN,0.0,0.0,NaN,0.0,0.0
1,16373956,58548660,1.0,1.0,NaN,NaN,NaN,NaN,NaN
2,11861017,58315601,NaN,NaN,1.0,NaN,NaN,NaN,1.0
3,19926301,58960487,NaN,NaN,1.0,1.0,NaN,NaN,NaN
4,18727840,50839615,NaN,1.0,-1.0,NaN,NaN,NaN,1.0


In [3]:
# Load MIMIC-IV patient information
import getpass
physionet_user = input("PhysioNet username/email: ")
physionet_pass = getpass.getpass("PhysioNet password: ")

PhysioNet username/email: BhavishyaGuntreddi
PhysioNet password: ··········


In [4]:
!wget -N -c --user={physionet_user} --password={physionet_pass} \
  -P {raw_path} \
  https://physionet.org/files/mimiciv/3.1/hosp/patients.csv.gz

patients = pd.read_csv(
    f'{raw_path}/patients.csv.gz'
)

print("Patients:", patients.shape)
patients.head()

--2026-08-24 11:53:04--  https://physionet.org/files/mimiciv/3.1/hosp/patients.csv.gz
Resolving physionet.org (physionet.org)... 18.25.8.254
Connecting to physionet.org (physionet.org)|18.25.8.254|:443... connected.
HTTP request sent, awaiting response... 401 Unauthorized
Authentication selected: Basic realm="PhysioNet", charset="UTF-8"
Reusing existing connection to physionet.org:443.
HTTP request sent, awaiting response... 304 Not Modified
File ‘/content/drive/MyDrive/dissertation_project/data/raw/patients.csv.gz’ not modified on server. Omitting download.

Patients: (364627, 6)


,subject_id,gender,anchor_age,anchor_year,anchor_year_group,dod
0,10000032,F,52,2180,2014 - 2016,2180-09-09
1,10000048,F,23,2126,2008 - 2010,NaN
2,10000058,F,33,2168,2020 - 2022,NaN
3,10000068,F,19,2160,2008 - 2010,NaN
4,10000084,M,72,2160,2017 - 2019,2161-02-13


In [5]:
# Load the admission information
admissions = pd.read_csv(
    f'{raw_path}/admissions.csv.gz'
)

print("Admissions:", admissions.shape)

admissions.head()

Admissions: (546028, 16)


,subject_id,hadm_id,admittime,dischtime,deathtime,admission_type,admit_provider_id,admission_location,discharge_location,insurance,language,marital_status,race,edregtime,edouttime,hospital_expire_flag
0,10000032,22595853,2180-05-06 22:23:00,2180-05-07 17:15:00,NaN,URGENT,P49AFC,TRANSFER FROM HOSPITAL,HOME,Medicaid,English,WIDOWED,WHITE,2180-05-06 19:17:00,2180-05-06 23:30:00,0
1,10000032,22841357,2180-06-26 18:27:00,2180-06-27 18:49:00,NaN,EW EMER.,P784FA,EMERGENCY ROOM,HOME,Medicaid,English,WIDOWED,WHITE,2180-06-26 15:54:00,2180-06-26 21:31:00,0
2,10000032,25742920,2180-08-05 23:44:00,2180-08-07 17:50:00,NaN,EW EMER.,P19UTS,EMERGENCY ROOM,HOSPICE,Medicaid,English,WIDOWED,WHITE,2180-08-05 20:58:00,2180-08-06 01:44:00,0
3,10000032,29079034,2180-07-23 12:35:00,2180-07-25 17:55:00,NaN,EW EMER.,P06OTX,EMERGENCY ROOM,HOME,Medicaid,English,WIDOWED,WHITE,2180-07-23 05:54:00,2180-07-23 14:00:00,0
4,10000068,25022803,2160-03-03 23:16:00,2160-03-04 06:26:00,NaN,EU OBSERVATION,P39NWO,EMERGENCY ROOM,NaN,NaN,English,SINGLE,WHITE,2160-03-03 21:55:00,2160-03-04 06:26:00,0


In [6]:
# Keep only required admission columns
admission_features = admissions[
    [
        'subject_id',
        'hadm_id',
        'admittime',
        'dischtime',
        'admission_type',
        'admission_location',
        'insurance',
        'marital_status',
        'race'
    ]
].copy()

print(admission_features.shape)

(546028, 9)


In [7]:
# Select patient demographics
patient_features = patients[
    [
        'subject_id',
        'gender',
        'anchor_age'
    ]
].copy()

patient_features.head()

,subject_id,gender,anchor_age
0,10000032,F,52
1,10000048,F,23
2,10000058,F,33
3,10000068,F,19
4,10000084,M,72


In [8]:
# Merge demographics and admission information
structured = admission_features.merge(
    patient_features,
    on='subject_id',
    how='left'
)

print("Structured admission data:", structured.shape)
structured.head()

Structured admission data: (546028, 11)


,subject_id,hadm_id,admittime,dischtime,admission_type,admission_location,insurance,marital_status,race,gender,anchor_age
0,10000032,22595853,2180-05-06 22:23:00,2180-05-07 17:15:00,URGENT,TRANSFER FROM HOSPITAL,Medicaid,WIDOWED,WHITE,F,52
1,10000032,22841357,2180-06-26 18:27:00,2180-06-27 18:49:00,EW EMER.,EMERGENCY ROOM,Medicaid,WIDOWED,WHITE,F,52
2,10000032,25742920,2180-08-05 23:44:00,2180-08-07 17:50:00,EW EMER.,EMERGENCY ROOM,Medicaid,WIDOWED,WHITE,F,52
3,10000032,29079034,2180-07-23 12:35:00,2180-07-25 17:55:00,EW EMER.,EMERGENCY ROOM,Medicaid,WIDOWED,WHITE,F,52
4,10000068,25022803,2160-03-03 23:16:00,2160-03-04 06:26:00,EU OBSERVATION,EMERGENCY ROOM,NaN,SINGLE,WHITE,F,19


In [9]:
# Calculate admission duration
structured['admittime'] = pd.to_datetime(structured['admittime'])
structured['dischtime'] = pd.to_datetime(structured['dischtime'])

structured['length_of_stay_hours'] = (
    structured['dischtime'] - structured['admittime']
).dt.total_seconds() / 3600

structured[['hadm_id', 'anchor_age', 'gender', 'length_of_stay_hours']].head()

,hadm_id,anchor_age,gender,length_of_stay_hours
0,22595853,52,F,18.866667
1,22841357,52,F,24.366667
2,25742920,52,F,42.100000
3,29079034,52,F,53.333333
4,25022803,19,F,7.166667


In [10]:
# Download laboratory item dictionary
!wget -N -c --user={physionet_user} --password={physionet_pass} \
  -P {raw_path} \
  https://physionet.org/files/mimiciv/3.1/hosp/d_labitems.csv.gz


lab_items = pd.read_csv(
    f'{raw_path}/d_labitems.csv.gz'
)

print("Lab item dictionary:", lab_items.shape)

lab_items.head()

--2026-08-24 11:53:13--  https://physionet.org/files/mimiciv/3.1/hosp/d_labitems.csv.gz
Resolving physionet.org (physionet.org)... 18.25.8.254
Connecting to physionet.org (physionet.org)|18.25.8.254|:443... connected.
HTTP request sent, awaiting response... 401 Unauthorized
Authentication selected: Basic realm="PhysioNet", charset="UTF-8"
Reusing existing connection to physionet.org:443.
HTTP request sent, awaiting response... 304 Not Modified
File ‘/content/drive/MyDrive/dissertation_project/data/raw/d_labitems.csv.gz’ not modified on server. Omitting download.

Lab item dictionary: (1650, 4)


,itemid,label,fluid,category
0,50801,Alveolar-arterial Gradient,Blood,Blood Gas
1,50802,Base Excess,Blood,Blood Gas
2,50803,"Calculated Bicarbonate, Whole Blood",Blood,Blood Gas
3,50804,Calculated Total CO2,Blood,Blood Gas
4,50805,Carboxyhemoglobin,Blood,Blood Gas


In [11]:
# Identify a small set of useful laboratory tests
target_labs = [
    'glucose',
    'creatinine',
    'sodium',
    'potassium',
    'hemoglobin',
    'white blood cells',
    'platelet count',
    'urea nitrogen'
]

lab_items[
    lab_items['label'].str.lower().isin(target_labs)
][['itemid', 'label', 'fluid']]

,itemid,label,fluid
7,50809,Glucose,Blood
9,50811,Hemoglobin,Blood
31,50833,Potassium,Other Body Fluid
110,50912,Creatinine,Blood
129,50931,Glucose,Blood
168,50971,Potassium,Blood
180,50983,Sodium,Blood
202,51006,Urea Nitrogen,Blood
407,51222,Hemoglobin,Blood
450,51265,Platelet Count,Blood


In [12]:
# Find matching lab item IDs
lab_keywords = [
    'glucose',
    'creatinine',
    'sodium',
    'potassium',
    'hemoglobin',
    'white blood cell',
    'platelet',
    'urea nitrogen'
]

pattern = '|'.join(lab_keywords)

selected_lab_items = lab_items[
    lab_items['label'].str.contains(
        pattern,
        case=False,
        na=False
    )
].copy()

print("Selected laboratory items:", selected_lab_items.shape)

selected_lab_items[
    ['itemid', 'label', 'fluid']
].head(30)

Selected laboratory items: (104, 4)


,itemid,label,fluid
4,50805,Carboxyhemoglobin,Blood
7,50809,Glucose,Blood
9,50811,Hemoglobin,Blood
12,50814,Methemoglobin,Blood
20,50822,"Potassium, Whole Blood",Blood
22,50824,"Sodium, Whole Blood",Blood
31,50833,Potassium,Other Body Fluid
32,50834,"Sodium, Body Fluid",Other Body Fluid
39,50841,"Creatinine, Ascites",Ascites
40,50842,"Glucose, Ascites",Ascites


In [13]:
# Prepare IDs
valid_hadm_ids = set(
    structured['hadm_id'].dropna().astype(int)
)

selected_itemids = set(
    selected_lab_items['itemid'].astype(int)
)

print("Required admissions:", len(valid_hadm_ids))
print("Required laboratory item IDs:", len(selected_itemids))

Required admissions: 546028
Required laboratory item IDs: 104


In [14]:
# Create a lookup from MIMIC item IDs to laboratory names

lab_lookup = (
    selected_lab_items[
        ['itemid', 'label']
    ]
    .drop_duplicates('itemid')
    .copy()
)

lab_lookup['lab_name'] = (
    lab_lookup['label']
    .str.lower()
    .str.replace(' ', '_')
)

lab_lookup = lab_lookup.set_index('itemid')['lab_name'].to_dict()

print("Number of selected lab item IDs:", len(lab_lookup))
print("\nExample mappings:")
print(list(lab_lookup.items())[:10])

Number of selected lab item IDs: 104

Example mappings:
[(50805, 'carboxyhemoglobin'), (50809, 'glucose'), (50811, 'hemoglobin'), (50814, 'methemoglobin'), (50822, 'potassium,_whole_blood'), (50824, 'sodium,_whole_blood'), (50833, 'potassium'), (50834, 'sodium,_body_fluid'), (50841, 'creatinine,_ascites'), (50842, 'glucose,_ascites')]


In [15]:
# Process the huge file in chunks
lab_file = f'{raw_path}/labevents.csv.gz'

# Use a set for fast filtering
valid_hadm_ids = set(
    structured['hadm_id']
    .dropna()
    .astype(int)
)

selected_itemids = set(
    lab_lookup.keys()
)

# Store only small aggregated results
lab_results = []

print("Starting laboratory processing...")
print("Relevant admissions:", len(valid_hadm_ids))
print("Relevant lab item IDs:", len(selected_itemids))

for i, chunk in enumerate(
    pd.read_csv(
        lab_file,
        compression='gzip',
        usecols=['hadm_id', 'itemid', 'valuenum'],
        chunksize=100000
    )
):

    # Keep only our selected admissions
    chunk = chunk[
        chunk['hadm_id'].isin(valid_hadm_ids)
    ]

    # Keep only selected laboratory tests
    chunk = chunk[
        chunk['itemid'].isin(selected_itemids)
    ]

    if len(chunk) == 0:
        continue

    # Remove missing numeric values
    chunk['valuenum'] = pd.to_numeric(
        chunk['valuenum'],
        errors='coerce'
    )

    chunk = chunk.dropna(
        subset=['valuenum']
    )

    if len(chunk) == 0:
        continue

    # Convert item ID to simple laboratory name
    chunk['lab_name'] = chunk['itemid'].map(
        lab_lookup
    )

    # Aggregate immediately
    aggregated = (
        chunk
        .groupby(['hadm_id', 'lab_name'])['valuenum']
        .mean()
        .reset_index()
    )

    lab_results.append(aggregated)

    # Delete chunk from memory
    del chunk

    if i % 10 == 0:
        print(
            f"Processed chunk {i} | "
            f"stored aggregated rows: {sum(len(x) for x in lab_results):,}"
        )

print("Finished processing.")

Starting laboratory processing...
Relevant admissions: 546028
Relevant lab item IDs: 104
Processed chunk 0 | stored aggregated rows: 2,577
Processed chunk 10 | stored aggregated rows: 25,399
Processed chunk 20 | stored aggregated rows: 48,105
Processed chunk 30 | stored aggregated rows: 70,661
Processed chunk 40 | stored aggregated rows: 93,650
Processed chunk 50 | stored aggregated rows: 118,014
Processed chunk 60 | stored aggregated rows: 141,272
Processed chunk 70 | stored aggregated rows: 164,162
Processed chunk 80 | stored aggregated rows: 186,937
Processed chunk 90 | stored aggregated rows: 211,073
Processed chunk 100 | stored aggregated rows: 233,368
Processed chunk 110 | stored aggregated rows: 257,969
Processed chunk 120 | stored aggregated rows: 281,148
Processed chunk 130 | stored aggregated rows: 304,746
Processed chunk 140 | stored aggregated rows: 326,824
Processed chunk 150 | stored aggregated rows: 350,414
Processed chunk 160 | stored aggregated rows: 372,885
Processed 

In [16]:
import os
size_mb = os.path.getsize(f"{raw_path}/labevents.csv.gz") / (1024**2)
print(f"{size_mb:.1f} MB (target: ~2400 MB)")

2472.8 MB (target: ~2400 MB)


In [17]:
# Combine only the aggregated results
lab_summary_long = pd.concat(
    lab_results,
    ignore_index=True
)

print(
    "Aggregated laboratory rows:",
    lab_summary_long.shape
)

lab_summary_long.head()

Aggregated laboratory rows: (3667799, 3)


,hadm_id,lab_name,valuenum
0,20010003.0,creatinine,1.660000
1,20010003.0,"creatinine,_urine",114.000000
2,20010003.0,glucose,177.200000
3,20010003.0,hemoglobin,10.114286
4,20010003.0,platelet_count,95.428571


In [18]:
# Create the final laboratory table
lab_summary = (
    lab_summary_long
    .groupby(
        ['hadm_id', 'lab_name']
    )['valuenum']
    .mean()
    .unstack()
    .reset_index()
)

print(
    "Final laboratory feature table:",
    lab_summary.shape
)

lab_summary.head()

Final laboratory feature table: (435053, 54)


lab_name,hadm_id,%_hemoglobin_a1c,24_hr_creatinine,"albumin/creatinine,_urine","amylase/creatinine_ratio,_urine",carboxyhemoglobin,creatinine,"creatinine,_ascites","creatinine,_body_fluid","creatinine,_joint_fluid",...,"sodium,_stool","sodium,_urine","sodium,_whole_blood",urea_nitrogen,"urea_nitrogen,_ascites","urea_nitrogen,_body_fluid","urea_nitrogen,_pleural","urea_nitrogen,_urine",urine_creatinine,white_blood_cells
0,20000019.0,NaN,NaN,NaN,NaN,NaN,1.066667,NaN,NaN,NaN,...,NaN,NaN,NaN,16.666667,NaN,NaN,NaN,NaN,NaN,9.125000
1,20000024.0,NaN,NaN,NaN,NaN,NaN,1.100000,NaN,NaN,NaN,...,NaN,NaN,NaN,28.000000,NaN,NaN,NaN,NaN,NaN,4.900000
2,20000034.0,NaN,NaN,NaN,NaN,NaN,2.350000,NaN,NaN,NaN,...,NaN,NaN,NaN,24.500000,NaN,NaN,NaN,NaN,NaN,8.750000
3,20000041.0,10.4,NaN,NaN,NaN,NaN,1.100000,NaN,NaN,NaN,...,NaN,NaN,NaN,19.666667,NaN,NaN,NaN,NaN,NaN,8.666667
4,20000045.0,NaN,NaN,NaN,NaN,NaN,0.437500,NaN,NaN,NaN,...,NaN,NaN,NaN,18.000000,NaN,NaN,NaN,NaN,NaN,6.487500


In [19]:
# Keep manageable laboratory features
wanted_features = [
    'glucose',
    'creatinine',
    'sodium',
    'potassium',
    'hemoglobin',
    'white_blood_cell_count',
    'platelet_count',
    'urea_nitrogen'
]

available_features = [
    col for col in wanted_features
    if col in lab_summary.columns
]

print("Available laboratory features:")
print(available_features)

lab_summary = lab_summary[
    ['hadm_id'] + available_features
]

Available laboratory features:
['glucose', 'creatinine', 'sodium', 'potassium', 'hemoglobin', 'platelet_count', 'urea_nitrogen']


In [20]:
# Use the confirmed study-to-admission relationship
study_links = pd.read_csv(
    f'{processed_path}/matched_final.csv'
)

study_links = study_links[
    ['study_id', 'subject_id', 'hadm_id']
].drop_duplicates()

print("Study-admission links:", study_links.shape)
study_links.head()

Study-admission links: (164287, 3)


,study_id,subject_id,hadm_id
0,56555713,18176683,25271020
1,58253335,11675773,20376160
2,58073210,14289658,21354213
3,53472404,17622718,22437813
4,54833293,12514721,27737785


In [21]:
# Merge everything
structured_final = (
    reference[
        [
            'subject_id',
            'study_id'
        ] + [
            col for col in reference.columns
            if col in [
                'No Finding',
                'Support Devices',
                'Pleural Effusion',
                'Lung Opacity',
                'Atelectasis',
                'Cardiomegaly',
                'Edema'
            ]
        ]
    ]
    .merge(
        study_links,
        on=['subject_id', 'study_id'],
        how='left'
    )
    .merge(
        structured,
        on=['subject_id', 'hadm_id'],
        how='left',
        suffixes=('', '_admission')
    )
    .merge(
        lab_summary,
        on='hadm_id',
        how='left'
    )
)

print("Final structured dataset:", structured_final.shape)
structured_final.head()

Final structured dataset: (2200, 27)


,subject_id,study_id,No Finding,Support Devices,Pleural Effusion,Lung Opacity,Atelectasis,Cardiomegaly,Edema,hadm_id,...,gender,anchor_age,length_of_stay_hours,glucose,creatinine,sodium,potassium,hemoglobin,platelet_count,urea_nitrogen
0,19963242,51118747,1.0,NaN,0.0,0.0,NaN,0.0,0.0,28497038,...,F,60,192.366667,130.600000,0.780000,138.600000,3.560000,12.616667,271.833333,11.000000
1,16373956,58548660,1.0,1.0,NaN,NaN,NaN,NaN,NaN,26776982,...,F,81,138.466667,103.000000,1.200000,139.000000,4.316667,11.240000,216.600000,26.166667
2,11861017,58315601,NaN,NaN,1.0,NaN,NaN,NaN,1.0,27162817,...,M,86,1249.216667,172.549296,0.848529,139.029851,4.289706,8.604688,410.218750,30.352941
3,19926301,58960487,NaN,NaN,1.0,1.0,NaN,NaN,NaN,24898520,...,M,71,8.100000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,18727840,50839615,NaN,1.0,-1.0,NaN,NaN,NaN,1.0,27366694,...,M,61,251.666667,130.750000,1.000000,135.750000,4.175000,10.650000,277.000000,18.250000


In [22]:
# Remove duplicate columns
structured_final = structured_final.loc[
    :,
    ~structured_final.columns.duplicated()
]

print(structured_final.shape)

(2200, 27)


In [23]:
# Select the final structured variables
feature_columns = [
    'anchor_age',
    'length_of_stay_hours',
    'gender',
    'admission_type',
    'insurance',
    'marital_status',
    'race'
] + available_features

label_columns = [
    'No Finding',
    'Support Devices',
    'Pleural Effusion',
    'Lung Opacity',
    'Atelectasis',
    'Cardiomegaly',
    'Edema'
]

final_columns = (
    ['subject_id', 'study_id', 'hadm_id']
    + feature_columns
    + label_columns
)

structured_final = structured_final[
    [col for col in final_columns if col in structured_final.columns]
]

print("Final columns:", structured_final.columns.tolist())

Final columns: ['subject_id', 'study_id', 'hadm_id', 'anchor_age', 'length_of_stay_hours', 'gender', 'admission_type', 'insurance', 'marital_status', 'race', 'glucose', 'creatinine', 'sodium', 'potassium', 'hemoglobin', 'platelet_count', 'urea_nitrogen', 'No Finding', 'Support Devices', 'Pleural Effusion', 'Lung Opacity', 'Atelectasis', 'Cardiomegaly', 'Edema']


In [24]:
# Check missing values
missing = (
    structured_final
    .isna()
    .sum()
    .sort_values(ascending=False)
)

missing[missing > 0]

,0
No Finding,1629
Lung Opacity,1531
Atelectasis,1493
Cardiomegaly,1433
Edema,1426
Support Devices,1336
Pleural Effusion,1194
hemoglobin,152
platelet_count,149
glucose,147


In [25]:
# Fill numerical missing values
numeric_features = [
    col for col in feature_columns
    if col in structured_final.columns
    and pd.api.types.is_numeric_dtype(
        structured_final[col]
    )
]

categorical_features = [
    col for col in feature_columns
    if col in structured_final.columns
    and col not in numeric_features
]

print("Numerical:", numeric_features)
print("Categorical:", categorical_features)

Numerical: ['anchor_age', 'length_of_stay_hours', 'glucose', 'creatinine', 'sodium', 'potassium', 'hemoglobin', 'platelet_count', 'urea_nitrogen']
Categorical: ['gender', 'admission_type', 'insurance', 'marital_status', 'race']


In [26]:
for col in numeric_features:
    structured_final[col] = structured_final[col].fillna(
        structured_final[col].median()
    )

for col in categorical_features:
    structured_final[col] = structured_final[col].fillna(
        'Unknown'
    )

In [27]:
# Encode categorical variables
structured_final = pd.get_dummies(
    structured_final,
    columns=categorical_features,
    drop_first=True,
    dtype=int
)

print("After encoding:", structured_final.shape)

After encoding: (2200, 66)


In [28]:
# Standardise numerical features
encoded_feature_columns = [
    col for col in structured_final.columns
    if col not in (
        ['subject_id', 'study_id', 'hadm_id']
        + label_columns
    )
]

numeric_encoded = [
    col for col in encoded_feature_columns
    if pd.api.types.is_numeric_dtype(
        structured_final[col]
    )
]

scaler = StandardScaler()

structured_final[numeric_encoded] = scaler.fit_transform(
    structured_final[numeric_encoded]
)

print("Numerical features standardised.")

Numerical features standardised.


In [29]:
print("Final dataset shape:", structured_final.shape)

print("\nNumber of studies:",
      structured_final['study_id'].nunique())

print("Number of patients:",
      structured_final['subject_id'].nunique())

print("\nMissing values:")
print(
    structured_final.isna().sum().sum()
)

structured_final.head()

Final dataset shape: (2200, 66)

Number of studies: 2200
Number of patients: 2014

Missing values:
10042


,subject_id,study_id,hadm_id,anchor_age,length_of_stay_hours,glucose,creatinine,sodium,potassium,hemoglobin,...,race_OTHER,race_PATIENT DECLINED TO ANSWER,race_PORTUGUESE,race_UNABLE TO OBTAIN,race_UNKNOWN,race_WHITE,race_WHITE - BRAZILIAN,race_WHITE - EASTERN EUROPEAN,race_WHITE - OTHER EUROPEAN,race_WHITE - RUSSIAN
0,19963242,51118747,28497038,-0.177389,-0.296402,-0.056758,-0.443790,0.150003,-1.386139,1.245555,...,-0.186567,-0.067574,-0.067574,-0.047727,-0.240192,-1.287247,-0.052295,-0.036953,-0.149348,-0.130789
1,16373956,58548660,26776982,1.100943,-0.420871,-0.664053,-0.115496,0.259359,0.620986,0.514157,...,-0.186567,-0.067574,-0.067574,-0.047727,-0.240192,0.776852,-0.052295,-0.036953,-0.149348,-0.130789
2,11861017,58315601,27162817,1.405308,2.144140,0.866271,-0.390224,0.267519,0.549470,-0.885936,...,-0.186567,-0.067574,-0.067574,-0.047727,-0.240192,0.776852,-0.052295,-0.036953,-0.149348,-0.130789
3,19926301,58960487,24898520,0.492213,-0.721922,-0.231601,-0.294159,0.035676,-0.086371,-0.180051,...,-0.186567,-0.067574,-0.067574,-0.047727,-0.240192,0.776852,-0.052295,-0.036953,-0.149348,-0.130789
4,18727840,50839615,27366694,-0.116516,-0.159463,-0.053457,-0.271826,-0.629157,0.245203,0.200701,...,-0.186567,-0.067574,-0.067574,-0.047727,-0.240192,0.776852,-0.052295,-0.036953,-0.149348,-0.130789


In [30]:
output_file = f'{processed_path}/structured_processed.csv'

structured_final.to_csv(
    output_file,
    index=False
)

print("Saved:", output_file)
print("Shape:", structured_final.shape)

Saved: /content/drive/MyDrive/dissertation_project/data/processed/structured_processed.csv
Shape: (2200, 66)
